In [ ]:
%%time
import subprocess
subprocess.run(['pip', 'install', 'onnxruntime', '--quiet'], check=True)

import gc
import os
import time
import warnings
from pathlib import Path

import cv2
import librosa
import numpy as np
import onnxruntime as ort
import pandas as pd

warnings.filterwarnings('ignore')
print('All imports successful.')

In [ ]:
# --- paths ---
SAMPLE_SUB_PATH    = '/kaggle/input/birdclef-2026/sample_submission.csv'
TEST_SOUNDSCAPE_DIR = '/kaggle/input/birdclef-2026/test_soundscapes/'
MODEL_DIR          = '/kaggle/input/birdclef2026-models/'
OUTPUT_PATH        = '/kaggle/working/submission.csv'

# --- audio / mel params (must match training exactly) ---
SR          = 32000
WINDOW_SEC  = 5
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512
F_MIN       = 20
F_MAX       = 16000
IMG_SIZE    = 224

# --- inference params ---
BATCH_SIZE          = 16
SMOOTH_WINDOW       = 3           # temporal rolling-mean window (windows)
SAFETY_MINUTES      = 75          # abort inference if elapsed > this
INTRA_OP_THREADS    = 4

print(f'Test soundscapes: {TEST_SOUNDSCAPE_DIR}')
print(f'Model directory:  {MODEL_DIR}')
print(f'Output:           {OUTPUT_PATH}')

In [ ]:
# Load species list in the EXACT column order from sample_submission.csv
sub_template = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sub_template.columns[1:])   # drop 'row_id'
N_CLASSES = len(SPECIES_LIST)

assert N_CLASSES == 234, f'Expected 234 species columns, got {N_CLASSES}'
print(f'Species list loaded: {N_CLASSES} classes')
print(f'First 5: {SPECIES_LIST[:5]}')
print(f'Last  5: {SPECIES_LIST[-5:]}')

In [ ]:
def load_onnx_models(model_dir: str) -> list:
    """Load all .onnx files from model_dir; return InferenceSession list."""
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = INTRA_OP_THREADS
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    sessions = []
    for p in sorted(Path(model_dir).glob('*.onnx')):
        sess = ort.InferenceSession(
            str(p),
            sess_options=opts,
            providers=['CPUExecutionProvider'],
        )
        sessions.append(sess)
        print(f'  Loaded: {p.name}')

    if not sessions:
        raise FileNotFoundError(f'No .onnx files found in {model_dir}')
    return sessions


print('Loading ONNX models...')
SESSIONS = load_onnx_models(MODEL_DIR)
print(f'\n{len(SESSIONS)} model(s) loaded.')

# --- warm-up: single dummy inference to initialise ONNX Runtime ---
dummy = np.zeros((1, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
for sess in SESSIONS:
    input_name = sess.get_inputs()[0].name
    out = sess.run(None, {input_name: dummy})[0]
    assert out.shape == (1, N_CLASSES), f'Unexpected output shape: {out.shape}'
print('Warm-up OK: output shape (1, 234) confirmed for all models.')

# --- timing estimate ---
t0 = time.perf_counter()
for _ in range(10):
    for sess in SESSIONS:
        input_name = sess.get_inputs()[0].name
        sess.run(None, {input_name: dummy})
ms_per_window = (time.perf_counter() - t0) / 10 * 1000
test_files = sorted(Path(TEST_SOUNDSCAPE_DIR).glob('*.ogg'))
n_test_windows = len(test_files) * 12   # 12 × 5 s per 60-s file
est_mel_ms = 30 * n_test_windows
est_infer_ms = ms_per_window * n_test_windows
est_total_min = (est_mel_ms + est_infer_ms) / 60_000
print(f'\nTest files found: {len(test_files)}')
print(f'ONNX forward pass: {ms_per_window:.1f} ms/window (all {len(SESSIONS)} model(s))')
print(f'Estimated total inference time: {est_total_min:.1f} min  (budget: {SAFETY_MINUTES} min)')
if est_total_min > SAFETY_MINUTES:
    print('WARNING: Estimated time exceeds safety limit — consider reducing to 1 model.')

In [ ]:
# Inline mel spectrogram function — no external imports from src/

def compute_mel(audio_5s: np.ndarray, sr: int = SR) -> np.ndarray:
    """
    Convert a 5-second mono waveform to a (3, 224, 224) float32 array.
    Parameters must be byte-identical to those used in training.
    """
    mel = librosa.feature.melspectrogram(
        y=audio_5s,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = mel_db / 80.0          # normalize to [-1, 1]
    mel_db = np.clip(mel_db, -1.0, 1.0)
    mel_resized = cv2.resize(mel_db, (IMG_SIZE, IMG_SIZE))
    mel_3ch = np.stack([mel_resized, mel_resized, mel_resized], axis=0)  # (3, H, W)
    return mel_3ch.astype(np.float32)


def extract_windows(audio: np.ndarray, sr: int = SR) -> list:
    """Split audio into non-overlapping 5-second windows."""
    window_samples = sr * WINDOW_SEC
    n_windows = int(np.floor(len(audio) / window_samples))
    windows = [audio[i * window_samples : (i + 1) * window_samples] for i in range(n_windows)]
    remainder = audio[n_windows * window_samples :]
    if len(remainder) > 0:
        pad = np.zeros(window_samples, dtype=audio.dtype)
        pad[:len(remainder)] = remainder
        windows.append(pad)
    return windows


def infer_batch(sessions: list, mel_batch: np.ndarray) -> np.ndarray:
    """Run mel_batch through each session; return averaged sigmoid probs."""
    all_probs = []
    for sess in sessions:
        input_name = sess.get_inputs()[0].name
        logits = sess.run(None, {input_name: mel_batch})[0]          # (B, 234)
        probs = 1.0 / (1.0 + np.exp(-logits.astype(np.float64)))     # numerically safe
        all_probs.append(probs.astype(np.float32))
    return np.mean(all_probs, axis=0).astype(np.float32)


def temporal_smooth(preds: np.ndarray, window: int = SMOOTH_WINDOW) -> np.ndarray:
    """Rolling mean of size `window` along the time axis."""
    if window <= 1 or preds.shape[0] <= 1:
        return preds
    kernel = np.ones(window) / window
    smoothed = np.apply_along_axis(
        lambda x: np.convolve(x, kernel, mode='same'),
        axis=0,
        arr=preds,
    )
    return smoothed.astype(np.float32)


def predict_soundscape(sessions: list, audio_path: str) -> dict:
    """
    Return dict: row_id -> (234,) float32 probability vector.
    row_id format: '{stem}_{end_time_s}'
    """
    audio, _ = librosa.load(audio_path, sr=SR, mono=True)
    windows = extract_windows(audio)
    n_win = len(windows)

    mel_list = [compute_mel(w) for w in windows]

    all_probs = []
    for i in range(0, n_win, BATCH_SIZE):
        batch = np.stack(mel_list[i : i + BATCH_SIZE], axis=0)
        probs = infer_batch(sessions, batch)
        all_probs.append(probs)

    preds = np.concatenate(all_probs, axis=0)  # (n_win, 234)
    preds = temporal_smooth(preds)

    stem = Path(audio_path).stem
    result = {}
    for i in range(n_win):
        end_time = (i + 1) * WINDOW_SEC
        row_id = f'{stem}_{end_time}'
        result[row_id] = preds[i]
    return result


print('Mel / inference helpers defined.')

# Quick sanity check on a synthetic waveform
dummy_audio = np.random.randn(SR * WINDOW_SEC).astype(np.float32) * 0.01
mel_test = compute_mel(dummy_audio)
assert mel_test.shape == (3, IMG_SIZE, IMG_SIZE), f'Bad mel shape: {mel_test.shape}'
assert mel_test.dtype == np.float32
print(f'compute_mel check: shape={mel_test.shape}, dtype={mel_test.dtype}, range=[{mel_test.min():.3f}, {mel_test.max():.3f}]')

In [ ]:
# --- Main inference loop ---

GLOBAL_START = time.time()

all_rows = {}   # row_id -> (234,) probability vector
n_files = len(test_files)
aborted = False

for file_idx, audio_path in enumerate(test_files):
    elapsed = (time.time() - GLOBAL_START) / 60.0

    # Safety abort: save partial submission if time is running out
    if elapsed > SAFETY_MINUTES:
        print(f'\n*** SAFETY ABORT at file {file_idx}/{n_files} ({elapsed:.1f} min elapsed) ***')
        print('Saving partial submission...')
        aborted = True
        break

    try:
        rows = predict_soundscape(SESSIONS, str(audio_path))
        all_rows.update(rows)
    except Exception as e:
        print(f'  ERROR processing {audio_path.name}: {e}')
        continue

    # Progress report every 50 files
    if (file_idx + 1) % 50 == 0 or (file_idx + 1) == n_files:
        rate = (file_idx + 1) / elapsed if elapsed > 0 else float('inf')
        remaining_files = n_files - (file_idx + 1)
        est_remaining = remaining_files / rate if rate > 0 else 0
        print(
            f'File {file_idx + 1:>4}/{n_files} | '
            f'elapsed {elapsed:.1f}m | '
            f'est remaining {est_remaining:.1f}m'
        )

total_elapsed = (time.time() - GLOBAL_START) / 60.0
status = 'PARTIAL' if aborted else 'COMPLETE'
print(f'\nInference {status}: {len(all_rows)} windows from {file_idx + (0 if aborted else 1)} files in {total_elapsed:.1f} min')

In [ ]:
# --- Assemble and save submission ---

# Build DataFrame from collected predictions
rows_df = pd.DataFrame.from_dict(all_rows, orient='index', columns=SPECIES_LIST)
rows_df.index.name = 'row_id'
rows_df = rows_df.reset_index()

# If we have a partial submission, fill missing row_ids with zeros
# (we do NOT fill with zeros — a partial submission is better than a wrong one;
# Kaggle evaluates only rows present in sample_submission.csv)

# Enforce exact column order from sample_submission
final_cols = ['row_id'] + SPECIES_LIST
rows_df = rows_df[final_cols]

# Clip to [0, 1] as a safety guard (sigmoid output should already be in range)
rows_df[SPECIES_LIST] = rows_df[SPECIES_LIST].clip(0.0, 1.0)

# --- Verification ---
assert list(rows_df.columns[1:]) == SPECIES_LIST, 'Column order mismatch!'
assert not rows_df.isnull().any().any(), 'NaN values found in submission!'
assert (rows_df[SPECIES_LIST].values >= 0.0).all(), 'Values below 0 found!'
assert (rows_df[SPECIES_LIST].values <= 1.0).all(), 'Values above 1 found!'

rows_df.to_csv(OUTPUT_PATH, index=False)
print(f'Submission saved: {OUTPUT_PATH}')
print(f'Shape: {rows_df.shape}  (rows × columns)')
print(f'All values in [0, 1]: OK')
print(f'Column order matches sample_submission.csv: OK')
print(f'No NaN values: OK')

In [ ]:
# --- Sanity checks ---

submission = pd.read_csv(OUTPUT_PATH)
species_cols = submission.columns[1:]

print('=== Submission Summary ===')
print(f'Total rows:     {len(submission):,}')
print(f'Total columns:  {len(submission.columns):,}  (1 row_id + {len(species_cols)} species)')
print(f'Value range:    [{submission[species_cols].values.min():.6f}, {submission[species_cols].values.max():.6f}]')
print()

# Top-10 most active species (highest mean prediction)
mean_preds = submission[species_cols].mean().sort_values(ascending=False)
print('Top-10 most active species (mean prediction):')
print(mean_preds.head(10).to_string())
print()

# Flag species with suspiciously high mean (> 0.5 would be unusual)
high_prior = mean_preds[mean_preds > 0.5]
if len(high_prior) > 0:
    print(f'WARNING: {len(high_prior)} species have mean prediction > 0.5 (review these):')
    print(high_prior.to_string())
else:
    print('No species with mean prediction > 0.5: looks reasonable.')
print()

# Flag species predicted constant 0.0 for all windows (catastrophic failure)
all_zero = (submission[species_cols] == 0.0).all()
zero_species = species_cols[all_zero].tolist()
if zero_species:
    print(f'WARNING: {len(zero_species)} species are predicted 0.0 for ALL windows (check model output):')
    print(zero_species)
else:
    print('No species with all-zero predictions: OK.')
print()

print('First 5 rows:')
display(submission.head())